# TI-Toolbox — Scripting Workflow

End-to-end TI stimulation pipeline: **preprocess → optimize → simulate → analyze → group stats → visualize**.

Run inside the SimNIBS container (`NOTEBOOK` alias) with a BIDS project mounted at `/mnt/`.

In [ ]:
from tit import get_path_manager, __version__
from tit.pre import run_pipeline
from tit.sim import SimulationConfig, Montage, run_simulation, load_montages, list_montage_names
from tit.opt import FlexConfig, ExConfig, run_flex_search, run_ex_search
from tit.analyzer import Analyzer, run_group_analysis
from tit.stats import (
    GroupComparisonConfig,
    CorrelationConfig,
    run_group_comparison,
    run_correlation,
)
from tit.blender import MontageConfig, VectorConfig, RegionConfig, run_montage, run_vectors, run_regions

pm = get_path_manager()
print(f"TI-Toolbox v{__version__}")
print(f"Project: {pm.project_dir}")
print(f"Subjects: {pm.list_simnibs_subjects()}")

In [ ]:
# ── Edit these for your project ──────────────────────────────────
SUBJECT = "ernie"
EEG_NET = "GSN-HydroCel-185.csv"
SIMULATION = "L_Insula"
TARGET_MNI = (-35.0, 5.0, 5.0)  # MNI coordinates of target ROI
ROI_RADIUS = 10.0                # mm

---
## 1. Preprocessing

Each flag toggles an independent step. Run only what you need.

In [ ]:
run_pipeline(
    subject_ids=[SUBJECT],
    convert_dicom=True,
    run_recon=True,
    create_m2m=True,
    run_tissue_analysis=True,
    # Diffusion (optional — requires DWI data):
    # run_qsiprep=True,
    # run_qsirecon=True,
    # extract_dti=True,
)

---
## 2. Optimize Electrode Placement

Two strategies: **flex search** (differential evolution on the scalp) and **exhaustive search** (brute-force over a candidate pool).

### 2a. Flex Search

In [ ]:
flex_config = FlexConfig(
    subject_id=SUBJECT,
    goal="mean",                     # "mean", "max", or "focality"
    postproc="max_TI",
    current_mA=2.0,
    electrode=FlexConfig.ElectrodeConfig(
        shape="ellipse",
        dimensions=[8.0, 8.0],
        gel_thickness=4.0,
    ),
    roi=FlexConfig.SphericalROI(
        x=TARGET_MNI[0], y=TARGET_MNI[1], z=TARGET_MNI[2],
        radius=ROI_RADIUS,
        use_mni=True,
    ),
    n_multistart=3,
    min_electrode_distance=5.0,
)

flex_result = run_flex_search(flex_config)
print(f"Best value: {flex_result.best_value:.4f}")
print(f"Output:     {flex_result.output_folder}")

### 2b. Exhaustive Search

Requires a pre-computed leadfield HDF5. Two electrode modes: **pool** (any electrode → any channel) or **bucket** (pre-assigned to channels).

In [ ]:
# Pool mode
ex_config = ExConfig(
    subject_id=SUBJECT,
    leadfield_hdf=f"{SUBJECT}_leadfield_EEG10-20_Okamoto_2004.hdf5",
    roi_name="L-Insula",
    electrodes=ExConfig.PoolElectrodes(
        electrodes=["Fp1", "Fp2", "C3", "C4", "Cz", "Pz", "T7", "T8"]
    ),
    total_current=2.0,
    current_step=0.2,
    channel_limit=1.2,
)

ex_result = run_ex_search(ex_config)
print(f"Combinations tested: {ex_result.n_combinations}")
print(f"Results CSV:         {ex_result.results_csv}")

---
## 3. Simulate

Run SimNIBS FEM and compute TI field quantities (TI_max, TI_normal, etc.).  
Simulation type is auto-detected: 2 pairs = TI, 4+ pairs = mTI.

In [ ]:
# Load a pre-defined montage or construct one inline
montages = load_montages(montage_names=[SIMULATION], eeg_net=EEG_NET)

# Or build explicitly:
# montages = [Montage(
#     name="Custom_TI",
#     mode=Montage.Mode.NET,
#     electrode_pairs=[("E030", "E020"), ("E095", "E070")],
#     eeg_net=EEG_NET,
# )]

sim_config = SimulationConfig(
    subject_id=SUBJECT,
    montages=montages,
    conductivity="scalar",           # "scalar", "vn", "dir", "mc"
    intensities=[1.0, 1.0],          # mA per pair
    electrode_shape="ellipse",
    electrode_dimensions=[8.0, 8.0],
    gel_thickness=4.0,
    rubber_thickness=2.0,
)

run_simulation(sim_config)

---
## 4. Analyze

Extract field statistics from a completed simulation — spherical or atlas-based ROI.

In [ ]:
analyzer = Analyzer(
    subject_id=SUBJECT,
    simulation=SIMULATION,
    space="mesh",        # "mesh" or "voxel"
)

# Spherical ROI
result = analyzer.analyze_sphere(
    center=TARGET_MNI,
    radius=ROI_RADIUS,
    coordinate_space="MNI",
    visualize=True,
)
print(f"ROI mean: {result.roi_mean:.4f} V/m")
print(f"ROI max:  {result.roi_max:.4f} V/m")
print(f"Focality: {result.roi_focality:.4f}")

In [ ]:
# Atlas-based ROI
cortical_result = analyzer.analyze_cortex(
    atlas="DK40",
    region="lh.insula",
    visualize=True,
)
print(f"ROI mean: {cortical_result.roi_mean:.4f} V/m")
print(f"ROI max:  {cortical_result.roi_max:.4f} V/m")

---
## 5. Group Statistics — Permutation Testing

Cluster-based permutation testing for group comparisons and brain-behavior correlations.

### 5a. Group Comparison

In [ ]:
comparison_config = GroupComparisonConfig(
    analysis_name="active_vs_sham",
    subjects=[
        GroupComparisonConfig.Subject(subject_id="101", simulation_name=SIMULATION, response=1),
        GroupComparisonConfig.Subject(subject_id="102", simulation_name=SIMULATION, response=1),
        GroupComparisonConfig.Subject(subject_id="103", simulation_name=SIMULATION, response=0),
        GroupComparisonConfig.Subject(subject_id="104", simulation_name=SIMULATION, response=0),
    ],
    test_type=GroupComparisonConfig.TestType.UNPAIRED,
    alternative=GroupComparisonConfig.Alternative.TWO_SIDED,
    cluster_stat=GroupComparisonConfig.ClusterStat.MASS,
    n_permutations=5000,
    tissue_type=GroupComparisonConfig.TissueType.GREY,
)

comp_result = run_group_comparison(comparison_config)
print(f"Significant clusters: {comp_result.n_significant_clusters}")
print(f"Significant voxels:   {comp_result.n_significant_voxels}")
print(f"Output:               {comp_result.output_dir}")

### 5b. Brain-Behavior Correlation

In [ ]:
correlation_config = CorrelationConfig(
    analysis_name="dose_response",
    subjects=[
        CorrelationConfig.Subject(subject_id="101", simulation_name=SIMULATION, effect_size=0.45),
        CorrelationConfig.Subject(subject_id="102", simulation_name=SIMULATION, effect_size=0.32),
        CorrelationConfig.Subject(subject_id="103", simulation_name=SIMULATION, effect_size=0.55),
        CorrelationConfig.Subject(subject_id="104", simulation_name=SIMULATION, effect_size=0.28),
    ],
    correlation_type=CorrelationConfig.CorrelationType.PEARSON,
    n_permutations=5000,
    tissue_type=CorrelationConfig.TissueType.GREY,
)

corr_result = run_correlation(correlation_config)
print(f"Significant clusters: {corr_result.n_significant_clusters}")
print(f"Output:               {corr_result.output_dir}")

---
## 6. Blender Visualization

Generate publication-ready 3D exports: electrode montage scenes, vector field arrows, and atlas-labeled cortical regions.

### 6a. Montage Publication Scene

In [ ]:
montage_result = run_montage(MontageConfig(
    subject_id=SUBJECT,
    simulation_name=SIMULATION,
    electrode_diameter_mm=10.0,
    electrode_height_mm=6.0,
    show_full_net=True,
))
print(f"Blend file: {montage_result.final_blend}")

### 6b. Vector Field Export

In [ ]:
run_vectors(VectorConfig(
    subject_id=SUBJECT,
    simulation_name=SIMULATION,
    export_ti_normal=True,
    count=50_000,
    color=VectorConfig.Color.MAGSCALE,
))

### 6c. Cortical Region Export

In [ ]:
n_exported = run_regions(RegionConfig(
    subject_id=SUBJECT,
    simulation_name=SIMULATION,
    format=RegionConfig.Format.PLY,
    atlas="DK40",
    field_name="TI_max",
    colormap="hot",
))
print(f"Exported {n_exported} cortical regions")